# BLACK × Kaggriculture — Deep Research Notebook
Notebook-first. Official environment → public top implementations → replay/leaderboard → opponent → market → geometry → paired A/B.
Phase3 production/task/plant/terminal = FREEZE. final cash and ladder μ are separate metrics.

In [ ]:
from pathlib import Path
import json,sys
ROOT=Path('artifacts/kaggriculture_black'); ROOT.mkdir(parents=True,exist_ok=True)
sys.path.append(str(Path('research').resolve()))
from kaggriculture_black_research import *
print(ROOT.resolve())

## 1. Official truth boundary
Verify the real 720-turn two-player observation contract and public opponent farm visibility. Never inspect opponent private shed/inventory.

In [ ]:
from kaggle_environments import make
env=make('kaggriculture',configuration={'episodeSteps':1},debug=False)
env.run(['starter','starter'])
obs=env.steps[0][0]['observation']
print('obs keys:',sorted(obs))
print('farms:',len(obs['farms']),'player:',obs['player'])
print('opponent public signature:',opp_signature(obs['farms'][1-int(obs['player'])]))
assert 'private' in obs


## 2. Parallel Kaggle artifact collection
Leaderboard, own submissions, and competition files are collected concurrently. Missing data is marked unavailable rather than inferred.

In [ ]:
results,manifest=execute_collection()
print({k:v['returncode'] for k,v in results.items()})
lb=json.loads((ROOT/'leaderboard_normalized.json').read_text()) if (ROOT/'leaderboard_normalized.json').exists() else []
print('top20:',lb[:20])

## 3. Public strategy inventory
Official source plus two public implementations are stored as provenance. GzmCR emphasizes market price/demand/inventory/cash/season timing; Deepesh emphasizes market-first and Manhattan nearest-target action economy. These are hypotheses, not proof of leaderboard strength.

In [ ]:
for s in SOURCE_MANIFEST: print(s['class'],s['url'],s['finding'])

## 4. User-observed leaderboard snapshot
Preserve the 2026-08-18 observed values as provenance; fresh Kaggle retrieval supersedes them when available.

In [ ]:
print(json.dumps(USER_LEADERBOARD_SNAPSHOT,ensure_ascii=False,indent=2))

## 5. Geometry / action economy
Measure move/work share, land expansion timing, hire timing, pickup/drop, market-order counts, and terminal liquidation. Geometry earns promotion only through paired evidence.

In [ ]:
def geometry_metrics(events):
 m={'move':0,'work':0,'market_orders':0,'land_buys':[],'hires':[],'pickup':0,'drop':0}
 for e in events:
  m['move']+=int(e.get('move',0)); m['work']+=int(e.get('work',0)); m['market_orders']+=int(e.get('market_orders',0))
  if e.get('land_buy') is not None: m['land_buys'].append(e['land_buy'])
  if e.get('hire') is not None: m['hires'].append(e['hire'])
  m['pickup']+=int(e.get('pickup',0)); m['drop']+=int(e.get('drop',0))
 total=m['move']+m['work']; m['move_share']=m['move']/total if total else 0
 return m
print('geometry scorer ready')

## 6. Opponent divergence — public only
Estimate supply pressure from `farms[1-player]`. Only SELL ordering may consume this signal; production/planting stays frozen.

In [ ]:
opp=obs['farms'][1-int(obs['player'])]
print(pressure_from_public_farm(opp))

## 7. Market pressure / liquidation timing
Separate opponent supply from town demand and price state. Fertilizer is explicitly tested as liquidation timing, not production volume.

In [ ]:
def market_pressure(prices,inventory,demand=None):
 demand=demand or {}; out={}
 for p in set(prices)|set(inventory)|set(demand):
  px=float(prices.get(p,0) or 0); inv=float(inventory.get(p,0) or 0); dem=float(demand.get(p,0) or 0)
  out[p]={'price':px,'inventory':inv,'demand':dem,'sell_pressure':inv/max(1,dem+1)}
 return out
print('market scorer ready')

## 8. Replay winner/loser mining
Normalize public replay events to day/hour/player/action/product/position/cash. Aggregate across replays. A single replay never promotes a mechanism.

In [ ]:
def paired_delta(winner,loser,key): return float(winner.get(key,0) or 0)-float(loser.get(key,0) or 0)
print('paired replay scorer ready')

## 9. Candidate matrix
A baseline; B sparse opponent SELL; C geometry; D market timing; E B+D; F B+C+D. Run same seeds and both seats. Reject runtime/contract regressions.

In [ ]:
CANDIDATES=['A_BASE','B_OPP_SELL','C_GEOMETRY','D_MARKET_TIMING','E_OPP+MARKET','F_OPP+GEOMETRY+MARKET']
for c in CANDIDATES: print(c)
print('seeds=',SEEDS)

## 10. Evidence ledger / final gate
Source classes are OFFICIAL / PUBLIC_CODE / MEASURED / INFERENCE. Promotion requires measured paired evidence; otherwise freeze and return to data mining.

In [ ]:
ledger=[
 evidence('Public opponent farm is visible','OFFICIAL','farms[1-player] probe','PASS'),
 evidence('Opponent SELL adapter is a candidate only','INFERENCE','public pressure model','HOLD'),
 evidence('Cash and ladder μ are distinct metrics','MEASURED','submission/replay audit','PASS')
]
(ROOT/'evidence_ledger.json').write_text(json.dumps(ledger,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(ledger,ensure_ascii=False,indent=2))

### Promotion rule
Do not submit because local cash rose alone. Official contract → regression → paired head-to-head → leaderboard validation. If the adapter loses, revert it and keep Phase3 as fallback.